# K-Means Clustering from Scratch — ANSWER KEY

## Objective
Implement K-Means clustering from scratch using NumPy.

Implemented:
1. Euclidean distance
2. Find closest centroid
3. Assign clusters
4. Compute new centroids
5. Cost function
6. One iteration
7. Complete K-Means algorithm
8. Visualize clusters
9. Elbow method

This is the instructor solution copy — every **YOUR CODE HERE** cell from the assignment is filled in below.

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Initialize Inputs

`X` contains the data points, `K` is the number of clusters, and `centroids` holds the initial centroid positions.

Inspect the shape before continuing.

In [ ]:
X = np.array([
    [1.0,1.0],
    [1.5,2.0],
    [3.0,4.0],
    [5.0,7.0],
    [3.5,5.0],
    [4.5,5.0],
    [3.5,4.5]
])

print("X shape:", X.shape)

In [ ]:
K = 2
centroids = np.array([[1.0,1.0],[5.0,7.0]])

## 3. Step 1 — Compute Euclidean Distance

$$
d(p, c) = \sqrt{\sum_{j=1}^{n}(p_j - c_j)^2}
$$

Given as the worked example.

In [ ]:
def compute_distance(point, centroid):
    distance = np.linalg.norm(point - centroid, ord=2)
    return distance

In [ ]:
point = np.array([3, 4])
centroid = np.array([1, 1])

print(compute_distance(point, centroid))

# 3.605551275463989

## 4. Step 2 — Find the Closest Centroid

### Solution
`np.linalg.norm(centroids - point, axis=1)` computes the distance from `point` to every row of `centroids` in one vectorized call; `np.argmin()` returns the index of the smallest one.

In [ ]:
def find_closest_centroid(point, centroids):
    distances = np.linalg.norm(centroids - point, axis=1)
    index = np.argmin(distances)

    return index

In [ ]:
print(find_closest_centroid(np.array([3,4]), centroids))

# 0  (closer to [1.0, 1.0] than to [5.0, 7.0])

## 5. Step 3 — Assign Clusters

### Solution
Loop over every row of `X`, call `find_closest_centroid()`, and collect the results.

In [ ]:
def assign_clusters(X, centroids):
    idx = []

    for point in X:
        cluster = find_closest_centroid(point, centroids)
        idx.append(cluster)

    return np.array(idx)

In [ ]:
idx = assign_clusters(X, centroids)
print(idx)

# [0 0 0 1 1 1 1]

## 6. Step 4 — Compute New Centroids

*For each cluster, select all the points assigned to that cluster, compute the average of their coordinates, and use that average as the new centroid.*

### Solution
For each cluster `k`, select the points where `idx == k` and take their mean along `axis=0`.

In [ ]:
def compute_centroids(X, idx, K):
    new_centroids = []

    for k in range(K):
        points = X[idx == k]
        centroid = np.mean(points, axis=0)
        new_centroids.append(centroid)

    return np.array(new_centroids)

In [ ]:
new_centroids = compute_centroids(X, idx, K)
print(new_centroids)

## 7. Step 5 — Compute the Cost

$$
J = \frac{1}{m}\sum_{i=1}^{m} \left\| x^{(i)} - \mu_{idx^{(i)}} \right\|^2
$$

### Solution
For every point `i`, look up its assigned centroid, add the squared distance to a running total, then divide by `m`.

In [ ]:
def compute_cost(X, idx, centroids):
    m = len(X)

    total = 0
    for i in range(m):
        centroid = centroids[idx[i]]
        total += np.sum((X[i] - centroid)**2)

    return total / m

In [ ]:
print(compute_cost(X, idx, new_centroids))

## 8. Step 6 — Run One Iteration

### Solution
Assign clusters, recompute centroids, then compute the resulting cost.

In [ ]:
def run_iteration(X, centroids):
    idx = assign_clusters(X, centroids)
    new_centroids = compute_centroids(X, idx, len(centroids))
    cost = compute_cost(X, idx, new_centroids)

    return idx, new_centroids, cost

In [ ]:
idx, new_centroids, cost = run_iteration(X, centroids)

print(idx)
print(new_centroids)
print(cost)

## 9. Step 7 — Complete K-Means

*If the centroids have stopped changing (or changed only by a negligible amount), the algorithm has converged. So, we terminate the loop using `break`.*

### Solution
Assign clusters, compute new centroids, stop if they've converged (`np.allclose`), otherwise update and continue.

In [ ]:
def run_kmeans(X, initial_centroids, max_iters=100):
    centroids = initial_centroids.copy()

    for i in range(max_iters):
        idx = assign_clusters(X, centroids)
        new_centroids = compute_centroids(X, idx, len(centroids))

        if np.allclose(new_centroids, centroids):
            break

        centroids = new_centroids

    return centroids, idx

In [ ]:
final_centroids, idx = run_kmeans(X, centroids)
print(final_centroids)
print(idx)

## 10. Step 8 — Visualize the Clusters

### Solution
Scatter the points colored by `idx`, mark centroids with a distinct red `X`.

In [ ]:
plt.figure(figsize=(7,5))

plt.scatter(X[:, 0], X[:, 1], c=idx, s=60, cmap='Blues')
plt.scatter(final_centroids[:, 0], final_centroids[:, 1], marker='X', color='red', s=200)

plt.xlabel("X")
plt.ylabel("Y")
plt.title("K-Means Clustering Result")
plt.show()

## 11. Step 9 — Elbow Method

### Solution
For each `K`, randomly select `K` unique points as initial centroids, run `run_kmeans()`, and record the final cost.

In [ ]:
costs = []

for K in range(2, 5):

    random_indices = np.random.choice(len(X), K, replace=False)
    initial = X[random_indices]

    final_centroids, idx = run_kmeans(X, initial, 200)

    cost = compute_cost(X, idx, final_centroids)
    costs.append(cost)

In [ ]:
print(costs)

In [ ]:
plt.plot(range(2, 5), costs, marker='o')
plt.xlabel("K")
plt.ylabel("Cost")
plt.title("Elbow Method")
plt.show()

# Test your understanding — Initialization Sensitivity

### Solution

In [ ]:
seeds = [0, 1, 2]

for seed in seeds:
    np.random.seed(seed)
    random_indices = np.random.choice(len(X), 2, replace=False)
    initial = X[random_indices]

    final_centroids, idx = run_kmeans(X, initial, 200)
    cost = compute_cost(X, idx, final_centroids)

    print(f"Seed {seed} -> centroids:\n{final_centroids}\ncost: {cost}\n")

**Answer:** Not necessarily. K-Means only guarantees convergence to a **local** minimum of the cost function, not the global one. On a small, well-separated dataset like this the different seeds usually settle on the same two clusters, but on larger or messier data, a bad initial spread of centroids (e.g. two centroids landing close together in the same region) can cause the algorithm to converge to a different — and worse — clustering. This sensitivity to initialization is exactly the motivation for smarter initialization schemes like K-Means++.

# Test your understanding — Reading the Elbow Plot

### Solution

**Answer:** For this dataset the cost drops sharply from `K=2` to `K=3` and then flattens out from `K=3` onward — `K=3` is the elbow. Cost keeps decreasing (even if only slightly) as `K` increases because adding more centroids can only ever help or leave the fit unchanged: with `K = m` (one centroid per point) the cost hits exactly zero. That's not a good reason to keep raising `K`, though, since more clusters than the data actually has starts fitting noise rather than real structure (overfitting) and makes the clustering harder to interpret — which is why the elbow, not the lowest cost, is the point to pick.